In [1]:
!pip install albumentations
!pip install ultralytics

In [2]:
import glob
from xml.etree import ElementTree as ET
import os
import shutil
from PIL import Image
from pathlib import Path
import pickle
import torchvision.transforms.v2


import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import ultralytics
from albumentations.pytorch.transforms import ToTensorV2
from matplotlib.patches import Rectangle
from torch import nn
from torchvision.models import ResNet50_Weights
from tqdm.notebook import tqdm


In [3]:
from dotenv import load_dotenv
import wandb

load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "gp5")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")

In [4]:
wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/evgeniy/.netrc
wandb: Currently logged in as: gigantina-ru (gigantina-ru-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Правильно раскидываем файлики по папочкам

In [5]:
'''source_dir = "dataset/train"
images_dir = os.path.join(source_dir, "images")
texts_dir = os.path.join(source_dir, "bboxes")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(texts_dir, exist_ok=True)

for filename in os.listdir(source_dir):
    filepath = os.path.join(source_dir, filename)
    ext = os.path.splitext(filename)[1].lower()

    if ext == ".jpg":
        shutil.move(filepath, os.path.join(images_dir, filename))
    elif ext == ".txt":
        shutil.move(filepath, os.path.join(texts_dir, filename))'''

'source_dir = "dataset/train"\nimages_dir = os.path.join(source_dir, "images")\ntexts_dir = os.path.join(source_dir, "bboxes")\n\nos.makedirs(images_dir, exist_ok=True)\nos.makedirs(texts_dir, exist_ok=True)\n\nfor filename in os.listdir(source_dir):\n    filepath = os.path.join(source_dir, filename)\n    ext = os.path.splitext(filename)[1].lower()\n\n    if ext == ".jpg":\n        shutil.move(filepath, os.path.join(images_dir, filename))\n    elif ext == ".txt":\n        shutil.move(filepath, os.path.join(texts_dir, filename))'

In [6]:
'''source_dir = "dataset/test"
images_dir = os.path.join(source_dir, "images")
texts_dir = os.path.join(source_dir, "bboxes")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(texts_dir, exist_ok=True)

for filename in os.listdir(source_dir):
    filepath = os.path.join(source_dir, filename)
    ext = os.path.splitext(filename)[1].lower()

    if ext == ".jpg":
        shutil.move(filepath, os.path.join(images_dir, filename))
    elif ext == ".txt":
        shutil.move(filepath, os.path.join(texts_dir, filename))'''

'source_dir = "dataset/test"\nimages_dir = os.path.join(source_dir, "images")\ntexts_dir = os.path.join(source_dir, "bboxes")\n\nos.makedirs(images_dir, exist_ok=True)\nos.makedirs(texts_dir, exist_ok=True)\n\nfor filename in os.listdir(source_dir):\n    filepath = os.path.join(source_dir, filename)\n    ext = os.path.splitext(filename)[1].lower()\n\n    if ext == ".jpg":\n        shutil.move(filepath, os.path.join(images_dir, filename))\n    elif ext == ".txt":\n        shutil.move(filepath, os.path.join(texts_dir, filename))'

In [7]:
new_labels = {
    0: 0,   # Bananas
    1: 1,   # Bananas (bag)
    2: 2,   # Blackberries
    3: 3,   # Raspberries
    # 4, 5 — Lemons — удалены
    6: 4,   # Grapes
    7: 5,   # Grapes (bag)
    8: 6,   # Tomatoes
    9: 7,   # Tomatoes (bag)
    10: 8,  # Apples
    11: 9,  # Apples (bag)
    # 12, 13 — Chilli — удалены
}

In [8]:
class_labels = {
    0: "Bananas",
    1: "Bananas",
    2: "Blackberries",
    3: "Raspberries",
    4: "Grapes",
    5: "Grapes",
    6: "Tomatoes",
    7: "Tomatoes",
    8: "Apples",
    9: "Apples",
}

C = 10

Напишем функцию, которая по изображению находит его описание в папке bboxes и переводит это описание в бодее удобные для работы метки.

В датасете они даны в формате "Класс Центр_прямоугольника_х, Центр_прямоугольника_y, Ширина, Высота". Мы хотим работать с библиотекой `albumentations`, там нужны метки в формате "Левый верхний_х, Левый верхний_y, Правый_нижний_х, Правый_нижний_y"

In [9]:
def get_yolo_data(image_path):
    image_path = Path(image_path)
    txt_path = str(image_path).replace("/images/", "/bboxes/").replace("jpg", "txt")
    img_w, img_h = Image.open(image_path).size

    bboxes = []
    with open(txt_path, "r") as f:
        for line in f:
            cls, xc, yc, w, h = map(float, line.split())
            cls = int(cls)

            if cls not in new_labels:
                continue

            xmin = int((xc - w / 2) * img_w)
            ymin = int((yc - h / 2) * img_h)
            xmax = int((xc + w / 2) * img_w)
            ymax = int((yc + h / 2) * img_h)

            bboxes.append([xmin, ymin, xmax, ymax, new_labels[cls]])

    return bboxes

Делаем класс, который будет хранить наш датасет

In [10]:
class PascalDataset(torch.utils.data.Dataset):
    def __init__(self, *, transform, root="dataset", mode="Train", seed=42):
        self.root = Path(root)
        self.transform = transform

        if mode == "Train":
            filenames = glob.glob(root + "/train/images/*")
        elif mode == "Test":
            filenames = glob.glob(root + "/test/images/*")

        # исключаем lemon и chilli
        self.filenames = np.array([f for f in filenames
                                  if "lemon" not in Path(f).stem.lower()
                                  and "chilli" not in Path(f).stem.lower()])
        
        np.random.seed(seed)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image = np.array(Image.open(fname))
        bboxes = get_yolo_data(fname)

        return self.transform(image=image, bboxes=bboxes)

    def __get_raw_item__(self, idx):
        fname = self.filenames[idx]
        return fname, get_yolo_data(fname)

    def __len__(self):
        return len(self.filenames)

Нормализуем и приведем к 512*512. Mean и std возьмем те, с которыми обучался ResNet, так как в будущем мы будем его использовать


In [11]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=dict(format="pascal_voc", min_visibility=0.3),
)

test_transform = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=dict(format="pascal_voc", min_visibility=0.5),
)

Создаем датасеты

In [12]:
train_ds = PascalDataset(root="dataset/", transform=train_transform, mode="Train")
test_ds = PascalDataset(root="dataset/", transform=test_transform, mode="Test")

Словарь, который переводит метку класса в название класса для визуализации. Одному классу принадлежит 2 метки, так как у фруктов есть две разновидности фотографий "в пакете" и "без пакета"

Функуция визуализации картинки с квадратиками

In [13]:
def visualize(images, bboxes):
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    fig, axes = plt.subplots(2, len(images) // 2 + len(images) % 2, figsize=(10, 8))

    for i, ax in enumerate(axes.reshape(-1)):

        ax.axis(False)

        if i >= len(images):
            break
        
        # денормализуем - возвращаем изображению изначальные цвета
        image = images[i]
        image = torch.permute(image, (1, 2, 0)).numpy()
        image = image * std + mean
        image = np.clip(image, 0, 1)

        ax.imshow(image)

        for bbox in bboxes[i]:
          # при показе готового фото из датасета с готовой разметкой, там нет уверенности в классе
          if len(bbox) == 5:
            xmin, ymin, xmax, ymax, cl = bbox
          # при показе прогноза, есть уверенность в классе
          if len(bbox) == 6:
            xmin, ymin, xmax, ymax, conf, cl = bbox
          rectangle = plt.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, fill=False, color="m")
          ax.add_patch(rectangle)
          ax.text(xmin, ymin-10, class_labels[cl], color="m", fontweight="bold")

    fig.tight_layout()
    plt.show()

Напишем функцию collate_fn, которая выполняет преобразование bounding boxes в формат карты признаков для детекции объектов.
Она делит изображение на 16 квадратов по вертикали и по горизонтали (в нашем датасете сторона квадрата по 32 пикселя).
Для каждого квадрата указываем 6 характеристик:
1. Относительный сдвиг центра bounding box относительно размера квадрата по Х (центр находится на 40% длины квадрата)
2. Относительный сдвиг центра bounding box относительно размера квадрата по Y (центр находится на 60% высоты квадрата)
3. Нормализованная ширина bounding box (он занимает 50% квадрата по ширине)
4. Нормализованная высота bounding box (он занимает 45% квадрата по высоте)
5. Confidence сетки - насколько мы уверены, что в этой клетке есть bbox
6. Класс детекции

![image](https://i.imgur.com/13YVxAd.jpeg)



In [14]:
def collate_fn(batch, pieces=(16, 16)):

    imgs = []
    batch_boxes = []

    for b in batch:
        imgs.append(b["image"])
        batch_boxes.append(b["bboxes"])

    imgs = torch.stack(imgs)
    b, c, h, w = imgs.shape

    if isinstance(pieces, int):
        pieces_h, pieces_w = pieces, pieces
    else:
        pieces_h, pieces_w = pieces

    ds_h = h // pieces_h
    ds_w = w // pieces_w

    target = imgs.new_zeros(b, 6, pieces_h, pieces_w)

    for i in range(len(batch_boxes)):
        boxes = imgs.new_tensor(batch_boxes[i])

        xmin, ymin, xmax, ymax, classes = boxes.T

        #считаем относительные w и h (3 и 4 каналы)
        w_box = (xmax-xmin)/w
        h_box = (ymax-ymin)/h

        #координаты центра в абсолютных значениях
        cx = (xmax + xmin) / 2
        cy = (ymax + ymin) / 2

        # считаем в какой квадрат попал центр bbox
        cx_idx = (cx // ds_w).long()
        cy_idx = (cy // ds_h).long()

        # считаем относительные сдвиги (1 и 2 каналы)
        cx_box = (cx - ds_w * cx_idx) / ds_w
        cy_box = (cy - ds_h * cy_idx) / ds_h

        # собираем каналы для одной клетки, conf пока равно 1
        target[i, :, cy_idx, cx_idx] = torch.stack(
            [cx_box, cy_box, w_box, h_box, torch.ones_like(cx_box), classes]
        )

    return {"image": imgs, "target": target}


## ПРОВЕСТИ ПЕРВЫЙ ЭКСПЕРИМЕНТ С САМОПИСНОЙ МОДЕЛЬЮ!!!!

Для начала напишем небольшую сверточную сеть для решения этой задачи

In [ ]:
# САМОПИСНЫЙ КЛАСС МОДЕЛЬКИ. 

Напишем функцию потерь аналогичную той, что была в начальных версиях YOLO 

Она состоит из 4-х основных компонентов:

1. localization loss - MSE по координатам бокса там, в боксе, где есть детектируемый объект (нашла ли модель объект в принципе)
2. box_loss - MSE от корней ширины и высоты bbox там, где есть детектируемый объект (как точно модель выделила объект рамочкой)
3. classification_loss - если детектируемый объект есть, то его кросс-энтропия по его классу (насколько правильно модель определила класс объекта)
4. confidence_loss - бинарная кросс-энтропия факта наличия объекта в квадрате. Учитываем верное угадывание, что объекта в квадрате нет, но с меньшим весом, чтобы модель не шла в сторону сильных ложноотрицательных результатов

In [15]:

def special_loss(pred, target, C=10):

    mask_obj = target[4] == 1

    loss_cx_box = torch.nn.functional.mse_loss(torch.masked_select(pred[0], mask_obj), torch.masked_select(target[0], mask_obj), reduction="sum")
    loss_cy_box = torch.nn.functional.mse_loss(torch.masked_select(pred[1], mask_obj), torch.masked_select(target[1], mask_obj), reduction="sum")
    localization_loss = loss_cx_box+loss_cy_box

    loss_w = torch.nn.functional.mse_loss(torch.sqrt(torch.masked_select(pred[2], mask_obj)), torch.sqrt(torch.masked_select(target[2], mask_obj)), reduction="sum")
    loss_h = torch.nn.functional.mse_loss(torch.sqrt(torch.masked_select(pred[3], mask_obj)), torch.sqrt(torch.masked_select(target[3], mask_obj)), reduction="sum")
    box_loss = loss_w+loss_h

    arr = []
    for i in range(5, 5 + C):
        arr.append(torch.masked_select(pred[i], mask_obj).unsqueeze(1))
    cls_pred = torch.cat(arr, dim=1) 
    cls_target = torch.masked_select(target[5], mask_obj).long()
    classification_loss = torch.nn.functional.cross_entropy(cls_pred, cls_target, reduction="sum")

    bce = nn.BCELoss(reduction="sum")
    yes_objects = bce(torch.masked_select(pred[4], mask_obj), torch.masked_select(target[4], mask_obj))
    no_objects = bce(torch.masked_select(pred[4], ~mask_obj), torch.masked_select(target[4], ~mask_obj))
    confidence_loss = yes_objects + 0.1*no_objects

    return localization_loss + box_loss + classification_loss + confidence_loss

In [16]:
loader = torch.utils.data.DataLoader(train_ds, 10, collate_fn=collate_fn, shuffle=True)

In [17]:
try:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.mps.is_available():
        device =torch.device("mps")
    else:
        device = torch.device("cpu")
except AttributeError:
    device = torch.device("cpu")

device

device(type='cpu')

In [ ]:
# ОБУЧЕНИЕ СУПЕР КРУТОЙ РУКОПИСНОЙ МОДЕЛЬКИ

In [ ]:
pickle.dump(model_test_1, open("model_test_1.pkl", 'wb'))

In [ ]:
with open('model_test_1.pkl', 'rb') as file:
    model1 = pickle.load(file)

Напишем функцию, которая переводит предсказания модели в удобные для рисования числа. Сразу на этом этапе учтем ситуацию, когда модель предсказывает один и тот же бокс в соседних квадратах. Будем использовать IoU, чтобы не отображать менее уверенные из пересекающихся боксов.

In [18]:
# считаем iou для боксов
def iou(bbox1, bbox2):
  xmin1, ymin1, xmax1, ymax1, conf1, cl1 = bbox1
  xmin2, ymin2, xmax2, ymax2, conf2, cl2 = bbox2
  intersection = torch.clamp(torch.min(xmax1, xmax2) - torch.max(xmin1, xmin2), min=0) * torch.clamp(torch.min(ymax1, ymax2) - torch.max(ymin1, ymin2), min=0)
  intersection = torch.clamp(intersection, min=0)
  union = (xmax1-xmin1)*(ymax1-ymin1) + (xmax2-xmin2)*(ymax2-ymin2) - intersection
  return intersection/union

# выкидываем менее уверенные боксы
def NMS(bboxes, threshold):
    leave = []
    classes = bboxes[:, 5].unique()

    for cls in classes:
        cls_bboxes = bboxes[bboxes[:, 5] == cls]
        cls_bboxes = cls_bboxes[torch.argsort(cls_bboxes[:, 4], descending=True)]

        while len(cls_bboxes) > 0:
            leave.append(cls_bboxes[0])
            left_check = []
            for box in cls_bboxes[1:]:
                if iou(cls_bboxes[0], box) <= threshold:
                    left_check.append(box)
                    
            if len(left_check) == 0:
                break
            cls_bboxes = torch.stack(left_check)

    if len(leave) == 0:
        return torch.zeros((0,6), dtype=torch.float32)

    return torch.stack(leave)


In [19]:
def decode_prediction_nms(pred, img_size=(512, 512), threshold=0.7):
    b, c, h, w = pred.shape
    img_w, img_h = img_size
    ds_h = img_h // h 
    ds_w = img_w // w
    
    cx_idx = torch.arange(w, dtype=torch.float32).reshape(1, 1, 1, w)
    cy_idx = torch.arange(h, dtype=torch.float32).reshape(1, 1, h, 1)
    result = []

    for i in range(len(pred)):
      image = pred[i]
      cx_box, cy_box, w_box, h_box, conf = image[:5]
      cl = torch.argmax(image[5:], dim=0)

      cx = cx_box*ds_w + ds_w*cx_idx
      cy = cy_box*ds_h + ds_h*cy_idx

      width = w_box * img_w
      height = h_box * img_h

      xmin = (cx - width / 2).squeeze()
      ymin = (cy - height / 2).squeeze()
      xmax = (cx + width / 2).squeeze()
      ymax = (cy + height / 2).squeeze()
      cl = cl.squeeze()

      res = torch.stack([xmin, ymin, xmax, ymax, conf, cl], dim=2)[conf.squeeze() > threshold]
      boxes_nms = NMS(res, threshold=0.6)

      result.append(boxes_nms.tolist())


    return result

Посмотрим на один батч

In [20]:
test_loader = torch.utils.data.DataLoader(test_ds, 10, collate_fn=collate_fn)
i = iter(test_loader)
batch = next(i)

In [ ]:
model_test_1.eval()
pred = model_test_1(batch["image"].to(device)).cpu()


In [ ]:
pred_decoded = decode_prediction_nms(pred, threshold=0.5)
visualize(batch["image"], pred_decoded)

In [ ]:
# ПОСЧИТАТЬ МЕТРИКИ

## Эксперимент 2

Проведем эксперимент с другой моделью, в этот раз возьмем слои от ResNet50, обучим исходные слои с меньшим шагом, а новые - с большим

In [21]:
class Detector(nn.Module):
    def __init__(self, C):
        super().__init__()
        model = torchvision.models.resnet50(weights=ResNet50_Weights.DEFAULT)
        self.rn = nn.Sequential(*list(model.children())[:-2])

        self.vgg = nn.Sequential(
           nn.Conv2d(in_channels=2048, out_channels=512, kernel_size=3, padding=1),
           nn.BatchNorm2d(num_features=512),
           nn.ReLU(),

           nn.Conv2d(in_channels=512, out_channels=128, kernel_size=3, padding=1),
           nn.BatchNorm2d(num_features=128),
           nn.ReLU(),

           nn.Conv2d(in_channels=128, out_channels=32, kernel_size=3, padding=1),
           nn.BatchNorm2d(num_features=32),
           nn.ReLU(),

           nn.Conv2d(in_channels=32, out_channels=5+C, kernel_size=3, padding=1),
           nn.BatchNorm2d(num_features=5+C),
           nn.Sigmoid(),
           
        )

    def forward(self, img):
        x = self.rn(img)
        x = self.vgg(x)
        return x

In [22]:
loader = torch.utils.data.DataLoader(train_ds, 10, collate_fn=collate_fn, shuffle=True)

In [23]:
try:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.mps.is_available():
        device =torch.device("mps")
    else:
        device = torch.device("cpu")
except AttributeError:
    device = torch.device("cpu")

device

device(type='cpu')

Обучаем модельку

In [ ]:
LR_BACKBONE=1e-5
LR_HEAD=1e-3
EPOCHS = 1
BATCH_SIZE = 1

config={
        "model": "Detector",
        "backbone": "ResNet50",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr_backbone": LR_BACKBONE,
        "lr_head": LR_HEAD,
        "seed": 21,
        "loss": "special_loss",
        "task": "detection",
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="cnn_detector_model_test_2", config=config)

In [ ]:
torch.manual_seed(21)
model_test_2 = Detector(C).to(device)



opt = torch.optim.Adam([
    {'params': model_test_2.rn.parameters(), 'lr': LR_BACKBONE},   
    {'params': model_test_2.vgg.parameters(), 'lr': LR_HEAD},
])


for e in tqdm(range(1, EPOCHS+1), desc=f"Epoch:"):
    epoch_losses = []

    for batch in tqdm(loader, desc=f"Epoch {e}", position=1, leave=False):
      batch_loss = 0
      images = batch["image"].to(device)
      targets = batch["target"].to(device)
      opt.zero_grad()
      pred = model_test_2(images)

      for el in range(pred.shape[0]):
            batch_loss += special_loss(pred[el], targets[el], C=C)

      batch_loss /= pred.shape[0]
      batch_loss.backward()
      opt.step()
      epoch_losses.append(batch_loss.item())


    mean_train_loss = np.mean(epoch_losses)

    print(f"Epoch {e} done; Train loss {mean_train_loss:.3f};")    

    wandb.log({"epoch": e, "train/loss": mean_train_loss})

Epoch::   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/270 [00:00<?, ?it/s]

Epoch 1 done; Train loss 24.502;


In [26]:
pickle.dump(model_test_2, open("models/model_test_2.pkl", 'wb'))

In [ ]:
artifact = wandb.Artifact(name="model_test_2", type="model",description="Тест логгирования модели Алины")

artifact.add_file("models/model_test_2.pkl")
run.log_artifact(artifact)

<Artifact model_test_2>

In [28]:
wandb.finish()

epoch,▁
train/loss,▁
epoch,1
train/loss,24.50236


In [ ]:
with open('model_test_2.pkl', 'rb') as file:
    model_test_2 = pickle.load(file)

In [ ]:
test_loader = torch.utils.data.DataLoader(test_ds, 10, collate_fn=collate_fn)
i = iter(test_loader)
batch = next(i)

In [ ]:
model_test_2.eval()
pred = model_test_2(batch["image"].to(device)).cpu()


In [ ]:
pred_decoded = decode_prediction_nms(pred, threshold=0.85)
visualize(batch["image"], pred_decoded)

In [ ]:
print(batch_loss.item())

## ПРОВЕСТИ ЭКСПЕРИМЕНТ С ПОЛНОЦЕННОЙ YOLO

ВОТ ЭТО ЗАГОТОВКА ДЛЯ ПОДСЧЕТА MAP

In [ ]:
!pip install torchmetrics

In [ ]:
def decode_prediction_modified(pred, upsample=32, threshold=0.7):
    decoded = decode_prediction_nms(pred, upsample=upsample, threshold=threshold)
    results = []
    for bbox in decoded:
        if len(bbox) == 0:
            results.append({
                "boxes": torch.zeros((0,4), dtype=torch.float32),
                "scores": torch.zeros((0,), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64)
            })
            continue

        bbox = torch.tensor(bbox, dtype=torch.float32)
        results.append({"boxes": bbox[:, :4], "scores": bbox[:, 4], "labels": bbox[:, 5].long()})

    return results


In [ ]:
from PIL import Image
from torchmetrics.detection.mean_ap import MeanAveragePrecision

model1.eval()
map_metric = MeanAveragePrecision(iou_type="bbox")

for batch in test_loader:
    images = batch["image"].to(device)
    targets = batch["target"]
    targets = decode_prediction_modified(targets, upsample=32, threshold=0.7)

    with torch.no_grad():
        pred = model1(batch["image"].to(device)).cpu()
        pred = decode_prediction_modified(pred, upsample=32, threshold=0.7)

    map_metric.update(pred, targets)

map_result = map_metric.compute()
print(map_result['map_50'])
